# Install anomalib


In [1]:
!pip install anomalib==0.7.0

  Preparing metadata (setup.py) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.8/136.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.1/569.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import shutil
from pathlib import Path

src = Path("/input/datasets/naimur978/machine-surface/Anomaly Detection Data/Feature 1")
dst = Path("/working/surface")
mask_src = src / "mask.png"

pairs = {
    "training": dst / "train" / "good",
    "test_ok": dst / "test" / "good",
    "test_nok": dst / "test" / "broken_large",
}

for folder, target in pairs.items():
    target.mkdir(parents=True, exist_ok=True)
    for f in (src / folder).iterdir():
        shutil.copy(f, target / f.name)

gt_target = dst / "ground_truth" / "broken_large"
gt_target.mkdir(parents=True, exist_ok=True)
for f in (src / "test_nok").iterdir():
    mask_name = f.stem + "_mask" + f.suffix
    shutil.copy(mask_src, gt_target / mask_name)

print("done:", dst)

done: /kaggle/working/surface


In [3]:
!git clone https://github.com/openvinotoolkit/anomalib.git
%cd anomalib
!git checkout df50c0b   

Cloning into 'anomalib'...
remote: Enumerating objects: 19541, done.
remote: Counting objects: 100% (348/348), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 19541 (delta 227), reused 124 (delta 117), pack-reused 19193 (from 3)
Receiving objects: 100% (19541/19541), 84.41 MiB | 36.30 MiB/s, done.
Resolving deltas: 100% (11535/11535), done.
/kaggle/working/anomalib
Note: switching to 'df50c0b'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at df50c0ba Fixed DSR (#1486)


# PaDim

In [ ]:
%%writefile /working/anomalib/src/anomalib/models/padim/config.yaml
dataset:
  name: mvtec #options: [mvtec, btech, folder]
  format: mvtec
  path: /working
  task: segmentation
  category: surface
  image_size: 256
  train_batch_size: 32
  test_batch_size: 1
  num_workers: 8
  transform_config:
    train: null
    val: null
  create_validation_set: false
  normalization: imagenet
  test_split_mode: from_dir # options: [from_dir, synthetic]
  test_split_ratio: 0.2 # fraction of train images held out testing (usage depends on test_split_mode)
  val_split_mode: same_as_test # options: [same_as_test, from_test, synthetic]
  val_split_ratio: 0.5 # fraction of train/test images held out for validation (usage depends on val_split_mode)
  tiling:
    apply: false
    tile_size: null
    stride: null
    remove_border_count: 0
    use_random_tiling: False
    random_tile_count: 16

model:
  name: padim
  backbone: resnet18
  pre_trained: true
  layers:
    - layer1
    - layer2
    - layer3
  normalization_method: min_max # options: [none, min_max, cdf]

metrics:
  image:
    - F1Score
    - AUROC
  pixel:
    - F1Score
    - AUROC
  threshold:
    method: adaptive #options: [adaptive, manual]
    manual_image: null
    manual_pixel: null

visualization:
  show_images: False # show images on the screen
  save_images: True # save images to the file system
  log_images: True # log images to the available loggers (if any)
  image_save_path: null # path to which images will be saved
  mode: full # options: ["full", "simple"]

project:
  seed: 42
  path: /working/results_padim_MVtec

logging:
  logger: [] # options: [comet, tensorboard, wandb, csv] or combinations.
  log_graph: false # Logs the model graph to respective logger.

optimization:
  export_mode: null # options: torch, onnx, openvino

# PL Trainer Args. Don't add extra parameter here.
trainer:
  enable_checkpointing: true
  default_root_dir: null
  gradient_clip_val: 0
  gradient_clip_algorithm: norm
  num_nodes: 1
  devices: 1
  enable_progress_bar: true
  overfit_batches: 0.0
  track_grad_norm: -1
  check_val_every_n_epoch: 1 # Don't validate before extracting features.
  fast_dev_run: false
  accumulate_grad_batches: 1
  max_epochs: 1
  min_epochs: null
  max_steps: -1
  min_steps: null
  max_time: null
  limit_train_batches: 1.0
  limit_val_batches: 1.0
  limit_test_batches: 1.0
  limit_predict_batches: 1.0
  val_check_interval: 1.0 # Don't validate before extracting features.
  log_every_n_steps: 50
  accelerator: auto # <"cpu", "gpu", "tpu", "ipu", "hpu", "auto">
  strategy: null
  sync_batchnorm: false
  precision: 32
  enable_model_summary: true
  num_sanity_val_steps: 0
  profiler: null
  benchmark: false
  deterministic: false
  reload_dataloaders_every_n_epochs: 0
  auto_lr_find: false
  replace_sampler_ddp: true
  detect_anomaly: false
  auto_scale_batch_size: false
  plugins: null
  move_metrics_to_cpu: false
  multiple_trainloader_mode: max_size_cycle


Overwriting /kaggle/working/anomalib/src/anomalib/models/padim/config.yaml


In [ ]:
!python /working/anomalib/tools/train.py --config /working/anomalib/src/anomalib/models/padim/config.yaml

/opt/conda/lib/python3.10/site-packages/anomalib/config/config.py:280: UserWarning: config.project.unique_dir is set to False. This does not ensure that your results will be written in an empty directory and you may overwrite files.
  warn(
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `PrecisionRecallCurve` will save all targets and predictions in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth
/opt/conda/lib/python3.10/site-packages/anomalib/utils/callbacks/__init__.py:142: UserWarning: Export option: None not found. Defaulting to no model export
  warnings.warn(f"Export option: {config.optimization.export_mode} not found. Defaulting to no model export")
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `

# PatchCore

In [ ]:
%%writefile /working/anomalib/src/anomalib/models/patchcore/config.yaml
dataset:
  name: mvtec
  format: mvtec
  path:  /working
  task: segmentation
  category: surface
  image_size: 256
  train_batch_size: 32
  test_batch_size: 1
  num_workers: 8
  transform_config:
    train: null
    val: null
  create_validation_set: false
  normalization: imagenet
  test_split_mode: from_dir
  test_split_ratio: 0.2
  val_split_mode: same_as_test
  val_split_ratio: 0.5
  tiling:
    apply: false
    tile_size: null
    stride: null
    remove_border_count: 0
    use_random_tiling: False
    random_tile_count: 16
model:
  name: patchcore
  backbone: wide_resnet50_2
  pre_trained: true
  layers:
    - layer2
    - layer3
  coreset_sampling_ratio: 0.1
  num_neighbors: 9
  normalization_method: min_max
metrics:
  image:
    - F1Score
    - AUROC
  pixel:
    - F1Score
    - AUROC
  threshold:
    method: adaptive
    manual_image: null
    manual_pixel: null
visualization:
  show_images: False
  save_images: True
  log_images: True
  image_save_path: null
  mode: full
project:
  seed: 42
  path: /working/results_patchcore_MVtec
logging:
  logger: []
  log_graph: false
optimization:
  export_mode: null
trainer:
  enable_checkpointing: true
  default_root_dir: null
  gradient_clip_val: 0
  gradient_clip_algorithm: norm
  num_nodes: 1
  devices: 1
  enable_progress_bar: true
  overfit_batches: 0.0
  track_grad_norm: -1
  check_val_every_n_epoch: 1
  fast_dev_run: false
  accumulate_grad_batches: 1
  max_epochs: 1
  min_epochs: null
  max_steps: -1
  min_steps: null
  max_time: null
  limit_train_batches: 1.0
  limit_val_batches: 1.0
  limit_test_batches: 1.0
  limit_predict_batches: 1.0
  val_check_interval: 1.0
  log_every_n_steps: 50
  accelerator: auto
  strategy: null
  sync_batchnorm: false
  precision: 32
  enable_model_summary: true
  num_sanity_val_steps: 0
  profiler: null
  benchmark: false
  deterministic: false
  reload_dataloaders_every_n_epochs: 0
  auto_lr_find: false
  replace_sampler_ddp: true
  detect_anomaly: false
  auto_scale_batch_size: false
  plugins: null
  move_metrics_to_cpu: false
  multiple_trainloader_mode: max_size_cycle

Overwriting /kaggle/working/anomalib/src/anomalib/models/patchcore/config.yaml


In [ ]:
!python  /working/anomalib/tools/train.py --config  /working/anomalib/src/anomalib/models/patchcore/config.yaml

/opt/conda/lib/python3.10/site-packages/anomalib/config/config.py:280: UserWarning: config.project.unique_dir is set to False. This does not ensure that your results will be written in an empty directory and you may overwrite files.
  warn(
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `PrecisionRecallCurve` will save all targets and predictions in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/wide_resnet50_racm-8234f177.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_racm-8234f177.pth
/opt/conda/lib/python3.10/site-packages/anomalib/utils/callbacks/__init__.py:142: UserWarning: Export option: None not found. Defaulting to no model export
  warnings.warn(f"Export option: {config.optimization.export_mode} not found. Defaulting to no model export")
/opt/conda/lib/python3.10/site-pa

# GANomaly

In [ ]:
%%writefile /working/anomalib/src/anomalib/models/ganomaly/config.yaml
dataset:
  name: mvtec
  format: mvtec
  path:  /working
  task: classification
  category: surface
  image_size: 256
  train_batch_size: 32
  test_batch_size: 32
  num_workers: 8
  transform_config:
    train: null
    val: null
  create_validation_set: false
  normalization: imagenet
  test_split_mode: from_dir
  test_split_ratio: 0.2
  val_split_mode: same_as_test
  val_split_ratio: 0.5
  tiling:
    apply: false
    tile_size: null
    stride: null
    remove_border_count: 0
    use_random_tiling: False
    random_tile_count: 16
model:
  name: ganomaly
  latent_vec_size: 100
  n_features: 64
  extra_layers: 0
  add_final_conv: true
  wadv: 1
  wcon: 50
  wenc: 1
  lr: 0.0002
  beta1: 0.5
  beta2: 0.999
  early_stopping:
    patience: 5
    metric: image_AUROC
    mode: max
  normalization_method: min_max
metrics:
  image:
    - F1Score
    - AUROC
  threshold:
    method: adaptive
    manual_image: null
    manual_pixel: null
visualization:
  show_images: False
  save_images: True
  log_images: True
  image_save_path: null
  mode: full
project:
  seed: 42
  path: /working/results_ganomaly_MVtec
logging:
  logger: []
  log_graph: false
optimization:
  export_mode: null
trainer:
  enable_checkpointing: true
  default_root_dir: null
  gradient_clip_val: 0
  gradient_clip_algorithm: norm
  num_nodes: 1
  devices: 1
  enable_progress_bar: true
  overfit_batches: 0.0
  track_grad_norm: -1
  check_val_every_n_epoch: 1
  fast_dev_run: false
  accumulate_grad_batches: 1
  max_epochs: 20
  min_epochs: null
  max_steps: -1
  min_steps: null
  max_time: null
  limit_train_batches: 1.0
  limit_val_batches: 1.0
  limit_test_batches: 1.0
  limit_predict_batches: 1.0
  val_check_interval: 1.0
  log_every_n_steps: 50
  accelerator: auto
  strategy: null
  sync_batchnorm: false
  precision: 32
  enable_model_summary: true
  num_sanity_val_steps: 0
  profiler: null
  benchmark: false
  deterministic: false
  reload_dataloaders_every_n_epochs: 0
  auto_lr_find: false
  replace_sampler_ddp: true
  detect_anomaly: false
  auto_scale_batch_size: false
  plugins: null
  move_metrics_to_cpu: false
  multiple_trainloader_mode: max_size_cycle

Overwriting /kaggle/working/anomalib/src/anomalib/models/ganomaly/config.yaml


In [ ]:
!python /working/anomalib/tools/train.py --config /working/anomalib/src/anomalib/models/ganomaly/config.yaml

/opt/conda/lib/python3.10/site-packages/anomalib/config/config.py:280: UserWarning: config.project.unique_dir is set to False. This does not ensure that your results will be written in an empty directory and you may overwrite files.
  warn(
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `PrecisionRecallCurve` will save all targets and predictions in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
/opt/conda/lib/python3.10/site-packages/anomalib/utils/callbacks/__init__.py:142: UserWarning: Export option: None not found. Defaulting to no model export
  warnings.warn(f"Export option: {config.optimization.export_mode} not found. Defaulting to no model export")
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `ROC` will save all targets and predictions in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*ar

# Autoencoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from pathlib import Path
from PIL import Image
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class ConvAutoencoder(nn.Module):
    def __init__(self, in_channels=3):
        super(ConvAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, in_channels, 4, stride=2, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

class AnomalyDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        for img_path in Path(image_dir).rglob('*'):
            if img_path.suffix.lower() in image_extensions:
                self.image_paths.append(str(img_path))
                label = 0 if 'good' in str(img_path).lower() else 1
                self.labels.append(label)
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

def ssim_map(img1, img2, window_size=11, C1=0.01**2, C2=0.03**2):
    channel = img1.size(1)
    kernel = torch.ones(channel, 1, window_size, window_size, device=img1.device) / (window_size ** 2)
    pad = window_size // 2
    mu1 = F.conv2d(img1, kernel, padding=pad, groups=channel)
    mu2 = F.conv2d(img2, kernel, padding=pad, groups=channel)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    sigma1_sq = F.conv2d(img1 * img1, kernel, padding=pad, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, kernel, padding=pad, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, kernel, padding=pad, groups=channel) - mu1_mu2
    ssim_val = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_val

def ssim_loss(img1, img2):
    return 1 - ssim_map(img1, img2).mean()

def train_autoencoder(model, train_loader, val_loader, epochs=50, lr=0.001, device='cuda'):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    mse_criterion = nn.MSELoss()
    model = model.to(device)
    best_val_loss = float('inf')
    patience = 8
    patience_counter = 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, _ in train_loader:
            images = images.to(device)
            reconstructed = model(images)
            loss = 0.5 * mse_criterion(reconstructed, images) + 0.5 * ssim_loss(reconstructed, images)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)
                reconstructed = model(images)
                loss = 0.5 * mse_criterion(reconstructed, images) + 0.5 * ssim_loss(reconstructed, images)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_autoencoder.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                model.load_state_dict(torch.load('best_autoencoder.pth'))
                break
    return model

def detect_anomalies(model, test_loader, device='cuda', threshold_percentile=90):
    model.eval()
    all_scores = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            reconstructed = model(images)
            smap = ssim_map(reconstructed, images)
            error = 1 - smap.view(smap.size(0), -1).mean(dim=1)
            all_scores.extend(error.cpu().numpy())
            all_labels.extend(labels.numpy())
    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    threshold = np.percentile(all_scores, threshold_percentile)
    predictions = (all_scores >= threshold).astype(int)
    return all_scores, all_labels, predictions, threshold

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

train_dataset = AnomalyDataset('/working/surface/train/good', transform=transform)
test_dataset = AnomalyDataset('/kaggle/working/surface/test', transform=transform)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_data, val_data = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

model = ConvAutoencoder(in_channels=3)
model = train_autoencoder(model, train_loader, val_loader, epochs=50, lr=0.001, device=device)

scores, labels, predictions, threshold = detect_anomalies(model, test_loader, device=device, threshold_percentile=90)

image_auroc = roc_auc_score(labels, scores)
image_f1 = f1_score(labels, predictions)
accuracy = np.mean(predictions == labels)

print(f"Image AUROC: {image_auroc:.4f}")
print(f"Image F1 Score: {image_f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Threshold: {threshold:.6f}")

Using device: cuda
Epoch 1/50 - Train Loss: 0.323733, Val Loss: 0.318569
Epoch 2/50 - Train Loss: 0.320833, Val Loss: 0.313053
Epoch 3/50 - Train Loss: 0.301591, Val Loss: 0.261934
Epoch 4/50 - Train Loss: 0.240557, Val Loss: 0.200751
Epoch 5/50 - Train Loss: 0.183397, Val Loss: 0.169000
Epoch 6/50 - Train Loss: 0.161681, Val Loss: 0.151299
Epoch 7/50 - Train Loss: 0.146486, Val Loss: 0.137298
Epoch 8/50 - Train Loss: 0.133797, Val Loss: 0.125210
Epoch 9/50 - Train Loss: 0.121908, Val Loss: 0.114581
Epoch 10/50 - Train Loss: 0.112169, Val Loss: 0.107000
Epoch 11/50 - Train Loss: 0.105023, Val Loss: 0.099441
Epoch 12/50 - Train Loss: 0.098363, Val Loss: 0.093738
Epoch 13/50 - Train Loss: 0.092477, Val Loss: 0.087938
Epoch 14/50 - Train Loss: 0.090966, Val Loss: 0.086207
Epoch 15/50 - Train Loss: 0.085893, Val Loss: 0.083564
Epoch 16/50 - Train Loss: 0.081648, Val Loss: 0.078087
Epoch 17/50 - Train Loss: 0.078322, Val Loss: 0.075112
Epoch 18/50 - Train Loss: 0.075181, Val Loss: 0.073114


In [11]:
import pandas as pd

all_results = {
    'PaDim': {'Image AUROC': 0.6875},
    'PatchCore': {'Image AUROC': 0.8750},
    'GANomaly': {'Image AUROC': 0.5417},
    'Autoencoder': {'Image AUROC': 0.3958}
}

results_df = pd.DataFrame(all_results).T
print(results_df.round(4))

             Image AUROC
PaDim             0.6875
PatchCore         0.8750
GANomaly          0.5417
Autoencoder       0.3958
